# 04. Phenome-wide 스캔 — 어떤 질환이 PPG 형태학에 흔적을 남기는가

**질문**: 특정 질환 하나가 아니라, 보유 데이터의 **모든 질환**에 대해
PPG 파형 형태학이 얼마나 신호를 담고 있는지 지도를 그린다.

**설계**
- 대상: ICD-10 3자리 코드 중 환자 ≥100명 → **156개**
- 특징: 43개 (형태학 20 + 박동간 변동성 20 + 검출률 3)
- 모델: 각 (질환 × 특징) 조합에 대해
  `표준화(feature) ~ disease + age + age² + sex + HR + 동반질환수`
- 대조군: 해당 코드가 **없는** 전체 환자 (표준 PheWAS 방식)
- 다중검정: 6,708건 전체에 BH-FDR

**왜 회귀 보정인가**: 156개 질환마다 1:2 매칭을 하면 계산량도 크고
질환별로 대조군이 달라져 비교가 어렵다. 회귀 보정이 PheWAS 표준이다.

In [ ]:
import os, sys, warnings, ast, collections
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore")

ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))
from ppg_fm.config import load

CFG  = load(); DATA = CFG["datasets"]["mimic_ext_ppg"]["root"]
INT  = os.path.join(ROOT, "data", "interim"); REP = os.path.join(ROOT, "reports")

## 1. 환자 단위 특징 테이블

`scripts/07_extract_all.py` 로 추출한 전 코호트 박동 특징을 환자 단위로 요약한다.
- 특징별 **중앙값** (대표값)
- 특징별 **박동간 변동성** IQR/median
- **검출률** — c–d파·RI·LVET 이 잡힌 박동 비율 (결측이 아니라 정보)

In [ ]:
B = pd.read_csv(os.path.join(INT, "beat_features_all.csv"), dtype={"subject": str})
print(f"박동 {len(B):,}  환자 {B.subject.nunique():,}")

FE = ["CT","LVET","CT_over_LVET","CT_over_IBI","LVET_over_IBI","dT","W25","W50","W75",
      "W50_over_IBI","RI","notch_rel_height","IPA","max_slope_norm","t_max_slope_rel",
      "b_over_a","c_over_a","d_over_a","e_over_a","aging_index"]

P = B.groupby("subject").agg({**{f:"median" for f in FE}, "IBI":"median",
                              "notch_found":"mean", "apg_found":"mean"})
P["HR"] = 60000 / P.IBI
P["n_beats"] = B.groupby("subject").size()
for col, name in [("d_over_a","cd_rate"), ("RI","ri_rate"), ("LVET","lvet_rate")]:
    P[name] = B.groupby("subject")[col].apply(lambda s: s.notna().mean())
cv = B.groupby("subject")[FE].agg(lambda s: (s.quantile(.75)-s.quantile(.25))/(abs(s.median())+1e-9))
cv.columns = [c+"_cv" for c in cv.columns]
P = P.join(cv).reset_index()
print(P.shape)

## 2. 환자 메타데이터와 ICD 매트릭스

In [ ]:
meta = (pd.read_csv(os.path.join(DATA, "metadata.csv"),
                    usecols=["subject_id","age","gender","icd10_truncated"], dtype=str)
          .drop_duplicates("subject_id").rename(columns={"subject_id":"subject"}))
meta["age"]  = pd.to_numeric(meta.age, errors="coerce")
meta["male"] = (meta.gender == "M").astype(int)

codes = {}
for r in meta.itertuples():
    try:    codes[r.subject] = set(ast.literal_eval(r.icd10_truncated))
    except Exception: codes[r.subject] = set()
meta["n_codes"] = meta.subject.map(lambda s: len(codes.get(s, ())))

P = P.merge(meta[["subject","age","male","n_codes"]], on="subject")
P = P[P.age.between(18, 95)]

cnt = collections.Counter()
for s in P.subject:
    for c in codes.get(s, ()): cnt[c] += 1
KEEP = [c for c, v in cnt.items() if v >= 100 and c != "NoD"]

Y = pd.DataFrame({c: [1 if c in codes.get(s, ()) else 0 for s in P.subject] for c in KEEP},
                 index=P.index)
print(f"환자 {len(P):,} · 분석 질환 {len(KEEP)}개")
P.to_csv(os.path.join(INT, "patient_features_all.csv"), index=False)

## 3. 회귀 스캔

각 특징을 **표준화**한 뒤 회귀하므로, disease 계수가 곧 표준화 효과크기(≈Cohen's d)가 된다.

공변량:
- `age`, `age²` — 연령은 파형의 최대 교란요인이며 비선형
- `male` — 성별
- `HR` — 모든 타이밍 지표에 직접 영향
- `n_codes` — **동반질환 부담**. ICU 환자는 코드가 많고, 이를 보정하지 않으면
  "아픈 환자일수록 파형이 다르다"는 자명한 결과를 질환 특이 신호로 오인한다.

In [ ]:
FEATS = [c for c in P.columns if c not in
         ("subject","IBI","n_beats","age","male","n_codes","HR","notch_found","apg_found")]

COV = P[["age","male","HR","n_codes"]].copy()
COV["age2"] = COV.age ** 2
X0 = sm.add_constant(COV.values.astype(float))

res = []
for f in FEATS:
    y  = P[f].values.astype(float)
    ok = np.isfinite(y) & np.isfinite(X0).all(1)
    if ok.sum() < 300: continue
    ys = (y[ok] - y[ok].mean()) / (y[ok].std() + 1e-12)   # 표준화
    Xc = X0[ok]
    for c in Y.columns:
        g  = Y[c].values[ok].astype(float)
        n1 = g.sum()
        if n1 < 80 or (len(g) - n1) < 80: continue
        try:
            m = sm.OLS(ys, np.column_stack([Xc, g])).fit()
            res.append((c, f, m.params[-1], m.pvalues[-1], int(n1)))
        except Exception:
            pass

R = pd.DataFrame(res, columns=["icd10","feat","beta","p","n_case"])
R["q"] = multipletests(R.p, method="fdr_bh")[1]
R.to_csv(os.path.join(REP, "phenome_wide.csv"), index=False)
print(f"검정 {len(R):,}건  ·  FDR<0.05 통과 {(R.q<0.05).sum():,} ({100*(R.q<0.05).mean():.1f}%)")

## 4. 질환별 신호 강도

어떤 질환이 PPG에 흔적을 많이 남기는가.

In [ ]:
dz = (R.groupby("icd10")
        .agg(n_sig=("q", lambda s: (s < .05).sum()),
             max_abs=("beta", lambda s: s.abs().max()),
             n_case=("n_case","first"))
        .sort_values(["n_sig","max_abs"], ascending=False))
dz.head(25)

## 5. 특징별 유용성

어떤 지표가 여러 질환에 걸쳐 유용한가.

In [ ]:
fz = (R.groupby("feat")
        .agg(n_sig=("q", lambda s: (s < .05).sum()),
             max_abs=("beta", lambda s: s.abs().max()))
        .sort_values("n_sig", ascending=False))
fz.head(20)

## 6. 개별 최강 연관

In [ ]:
R.reindex(R.beta.abs().sort_values(ascending=False).index).head(25)[
    ["icd10","feat","beta","q","n_case"]].round(4)

## 7. ⭐ `cd_rate` 의 양방향성

`cd_rate`(2차 미분 c–d파 검출률)가 가장 유용한 단일 특징이다.
주목할 점은 **방향이 질환군에 따라 갈린다**는 것이다.

In [ ]:
cd = R[(R.feat == "cd_rate") & (R.q < .05)].sort_values("beta")
print("=== c-d파가 사라지는 질환 (β < 0) ===")
print(cd[cd.beta < 0].head(10)[["icd10","beta","n_case"]].round(3).to_string(index=False))
print("\n=== c-d파가 보존되는 질환 (β > 0) ===")
print(cd[cd.beta > 0].tail(10)[["icd10","beta","n_case"]].round(3).to_string(index=False))

### 해석

| 방향 | 질환군 | 생리학적 해석 |
|---|---|---|
| **감소** (β<0) | 대동맥판막(I35), 심근병증(I42), 승모판(I34), 협심증(I20), 만성 허혈성(I25), 심부전(I50) | 박출 장애·동맥 경직 → 반사파 구조가 뭉개져 c–d 쌍이 융합 |
| **증가** (β>0) | 식도정맥류(I85), 알코올성 간질환(K70), 만성간염(B18), 간섬유화(K74), 복수(R18) | 문맥압항진 → 말초혈관확장·과역동 순환(낮은 SVR) → 반사파 구조 보존 |

**이 양방향성이 중요한 이유**: 만약 `cd_rate` 가 단순히 "아픈 정도"나 신호품질을
반영한다면 모든 질환에서 같은 방향으로 움직여야 한다. 실제로는 **혈관 긴장도의
방향에 따라 갈린다** — 이는 이 지표가 진짜 혈역학적 상태를 측정하고 있다는
강력한 방증이다.

## 8. 한계

| # | 한계 |
|---|---|
| 1 | ICD 코드 = 청구 코드, 심초음파 확진 아님 → 오분류. EchoNext로 검증 가능 |
| 2 | ICU 코호트 — 승압제·진정·기계환기가 파형에 직접 영향. 약물 정보 미보정 |
| 3 | 동반질환 클러스터 — `n_codes` 보정으로 부분적으로만 통제됨 |
| 4 | 단면 연관이며 인과 아님 |
| 5 | 단일 코호트, 외부검증 없음 (VitalDB·MIMIC-IV 필요) |
| 6 | 3자리 코드 — I35는 협착과 폐쇄부전을 합침 (혈역학 정반대) |